In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import itertools

# Summary
Plot all tuning of SPMD-MC and Qlearn on GridWorld (fromly GridWorld-loop).

Thus, `2026_04_10` will only be used to tune SPMD and Q-learn on Garnet.

## Tuning fixed MC-SPMD on GridWorld ~GridWorld-loop~

We will get this data from `2026_04_15/exp_0`

In [ ]:
exp_date = "2026_04_15"
n_exp = 36

Different values to query:
- `true value`: not practical and gives incorrect performances
- `point value`: may miss the point value in last iteration
- `agg value` suffers similar issue as point value, may assign too good of a score if state is missed.

**Heuristic**: we will access the last non-zero value.

In [ ]:
score_1 = np.zeros(n_exp*3)
root = os.path.join("..", "logs")
for i in range(len(score_1)):
    df = pd.read_csv(os.path.join(root, exp_date, "exp_0", "run_%d" % i, "seed=0.csv"))
    # score_1[i] = df['true value'].iloc[-1]
    # score_1[i] = df['point value'].iloc[-1]
    temp = df['point value'].to_numpy()
    score_1[i] = temp[np.nonzero(temp)[0][-1]] if np.any(temp) else np.inf

Now we will read off the hyperparameter settings and score.

In [ ]:
env_name_arr = [
    "gridworld_tiny", 
    "gridworld_small", 
    "gridworld_large", 
]
gamma_arr = [0.9, 0.99, 0.995]
T_mc_arr = [400, 2_000, 10_000, 50_000]
eta_arr = [5e-3, 2e-2, 5e-1]

exp_metadata = ["Exp id", "Env name", "gamma", "T_mc", "eta", "score", "best"]
row_format ="{:>10}|{:>25}|{:>10}|{:>10}|{:>10}|{:>15}|{:>5}"

print("")
print(row_format.format(*exp_metadata))
print("-" * (85+len(exp_metadata)-1))
group_size = 12

for ct, (env_name, gamma, T_mc, eta) in enumerate(itertools.product(env_name_arr, gamma_arr, T_mc_arr, eta_arr)):
    group_best = np.min(score_1[group_size*(ct//group_size):group_size*(1+ct//group_size)])
    is_group_best = group_best == score_1[ct]
    print(row_format.format(ct, env_name, gamma, T_mc, eta, score_1[ct], "***" if is_group_best else ""))

## Tuning fixed Q-learn on GridWorld-tune

We will get this data from `2026_04_16/exp_0`

In [ ]:
score_2 = np.zeros(27)
root = os.path.join("..", "logs")
for i in range(len(score_2)):
    df = pd.read_csv(os.path.join(root, "2026_04_16", "exp_0", "run_%d" % i, "seed=0.csv"))
    # score_2[i] = df['true value'].iloc[-1]
    temp = df['point value'].to_numpy()
    score_2[i] = temp[np.nonzero(temp)[0][-1]] if np.any(temp) else np.inf

In [ ]:
env_name_arr = [
    "gridworld_tiny", 
    "gridworld_small", 
    "gridworld_large", 
]
gamma_arr = [0.9, 0.99, 0.995]
total_samples = 100_000
alpha_arr = [-1, 1e-2, 1./total_samples]

exp_metadata = ["Exp id", "Env name", "gamma", "alpha", "score", "best"]
row_format ="{:>10}|{:>25}|{:>10}|{:>10}|{:>15}|{:>5}"

print("")
print(row_format.format(*exp_metadata))
print("-" * (75+len(exp_metadata)-1))
group_size = 3

for ct, (env_name, gamma, alpha) in enumerate(itertools.product(env_name_arr, gamma_arr, alpha_arr)):
    group_best = np.min(score_2[group_size*(ct//group_size):group_size*(1+ct//group_size)])
    is_group_best = group_best == score_2[ct]
    print(row_format.format(ct, env_name, gamma, alpha, score_2[ct], "***" if is_group_best else ""))